<a id="inizio"></a>
# 07 · Pandas: operazioni sui DataFrame · Soluzioni

⏱ ~70 min &nbsp;·&nbsp; Dati: `../Dati/U.S. Electricity Prices.csv`

**In questo notebook impariamo a**

- selezionare righe e colonne con `loc`, `iloc` e le condizioni
- gestire valori mancanti e duplicati, creare colonne nuove e ordinare
- raggruppare con `groupby` e unire DataFrame con `merge` e `concat`

<a id="indice"></a>
**Indice**

- [1. Selezione dei dati](#sez-1)
    - [1.1 Selezione di righe e colonne](#sez-1-1)
    - [1.2 Selezione condizionale](#sez-1-2)
- [2. Manipolazione dei dati](#sez-2)
    - [2.1 Valori mancanti](#sez-2-1)
    - [2.2 Duplicati](#sez-2-2)
    - [2.3 Trasformazione](#sez-2-3)
    - [2.4 Gestione degli indici](#sez-2-4)
    - [2.5 Rinomina delle colonne](#sez-2-5)
- [3. Ordinamento dei dati](#sez-3)
    - [3.1 sort_values e sort_index](#sez-3-1)
    - [3.2 Ordinamento su più colonne](#sez-3-2)
- [4. Groupby](#sez-4)
- [5. Unione e concatenazione di DataFrame](#sez-5)
    - [5.1 Merge](#sez-5-1)
    - [5.2 Concatenazione](#sez-5-2)
- [6. Esercizi](#sez-6)
    - [Esercizio 7.1 · Gli ascolti di un utente](#es-7-1)
    - [Esercizio 7.2 · Totale di ascolti per canzone](#es-7-2)
    - [Esercizio 7.3 · L'artista più ascoltato](#es-7-3)
    - [Esercizio 7.4 · Prezzo medio dell'elettricità per stato (facoltativo)](#es-7-4)

Importiamo le librerie e ricreiamo il DataFrame del notebook precedente: tre persone con nome,
età e città, e un indice con etichette.

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
# creare un DataFrame da un dizionario
dati = {
    "Nome": ["Alice", "Bob", "Charlie"],
    "Età": [24, 27, 22],
    "Città": ["Roma", "Milano", "Torino"],
}
df = pd.DataFrame(dati, index=["id_1", "id_2", "id_3"])
df

<a id="sez-1"></a>
## 1. Selezione dei dati

In pandas possiamo selezionare i dati per nome di colonna, per etichetta dell'indice o per
posizione.

<a id="sez-1-1"></a>
### 1.1 Selezione di righe e colonne

Possiamo selezionare una o più colonne di un DataFrame usando il nome della colonna come chiave.

In [ ]:
# selezionare una singola colonna
df[["Età"]]

In [ ]:
# selezionare più colonne
df[["Nome", "Città"]]

Con `loc` e `iloc` selezioniamo righe specifiche in base all'indice o alla posizione.

In [ ]:
# loc: in base all'indice
df.loc["id_1":"id_2", ["Età"]]

In [ ]:
# iloc: in base alla posizione
df.iloc[0:2, [1]]

Con `loc` lo slicing include anche l'ultima etichetta, con `iloc` l'ultima posizione è esclusa.

<a id="sez-1-2"></a>
### 1.2 Selezione condizionale

Possiamo filtrare i dati con condizioni booleane. La condizione da sola restituisce una Series
di `True` e `False`, una per riga.

In [ ]:
df["Età"] > 23

Messa tra parentesi quadre, la condizione tiene solo le righe con `True`.

In [ ]:
df[df["Età"] > 23]

Più condizioni si combinano con `&` (e), `|` (o) e `~` (non), ognuna tra parentesi tonde.

In [ ]:
condiz_eta = (df["Età"] > 23) & (df["Età"] < 26)
condiz_eta

In [ ]:
# righe con età tra 23 e 26 (esclusi), solo la colonna Nome
df.loc[(df["Età"] > 23) & (df["Età"] < 26), ["Nome"]]

Queste Series di `True` e `False` si chiamano maschere booleane.

<a id="sez-2"></a>
## 2. Manipolazione dei dati

La manipolazione dei dati in pandas comprende la gestione dei valori mancanti, la pulizia dei
dati e la trasformazione.

<a id="sez-2-1"></a>
### 2.1 Valori mancanti

I valori mancanti possono causare problemi nelle analisi e vanno gestiti.

In [ ]:
# creare un DataFrame con valori mancanti
dati_mancanti = {
    "A": [1, 2, np.nan],
    "B": [4, None, 6],
    "C": [7, 8, np.nan],
}
df_mancanti = pd.DataFrame(dati_mancanti)
df_mancanti

`isnull()` (e il suo opposto `notnull()`) dice per ogni cella se il valore manca. Con `.sum()`
contiamo i mancanti per colonna.

In [ ]:
# identificare i valori mancanti
df_mancanti.isnull().sum()

Per risolvere i valori mancanti abbiamo tre strade:

- **Riempimento**: `fillna()` sostituisce i valori mancanti con un valore specificato.
- **Rimozione**: `dropna()` elimina le righe o le colonne con valori mancanti.
- **Interpolazione**: stima i valori mancanti basandosi sui dati esistenti.

In [ ]:
# riempire i valori mancanti con zero
df_riempito = df_mancanti.fillna(0)
df_riempito

`fillna` restituisce una copia: per modificare la colonna la riassegniamo.

In [ ]:
df_mancanti["A"] = df_mancanti["A"].fillna(df_mancanti["A"].median())
df_mancanti.dropna()

- **Quando rimuovere**: se i dati mancanti sono pochi e la rimozione non influisce sull'analisi.
- **Quando riempire**: se i dati mancanti sono molti e c'è un valore appropriato per sostituirli.

<a id="sez-2-2"></a>
### 2.2 Duplicati

I duplicati possono distorcere i risultati delle analisi e vanno identificati e gestiti.

In [ ]:
# creare un DataFrame con duplicati
dati_doppi = {
    "Nome": ["Alice", "Bob", "Alice"],
    "Età": [24, 27, 24],
    "Città": ["Roma", "Milano", "Roma"],
}
df_doppi = pd.DataFrame(dati_doppi)
df_doppi

In [ ]:
# identificare i duplicati
df_doppi.duplicated()

In [ ]:
# rimuovere i duplicati
df_senza_doppi = df_doppi.drop_duplicates()
df_senza_doppi

<a id="sez-2-3"></a>
### 2.3 Trasformazione

Possiamo aggiungere, modificare o eliminare colonne in un DataFrame per adattarlo alle nostre
esigenze. Una colonna nuova si crea assegnando un'espressione a un nome che non esiste ancora.

In [ ]:
df["Anni alla pensione"] = 70 - df["Età"]
df

Quando la colonna nuova non viene da un conto, `.map` traduce ogni valore con un dizionario e
`.apply` applica una funzione a ogni valore.

In [ ]:
regioni = {"Roma": "Lazio", "Milano": "Lombardia", "Torino": "Piemonte"}
df["Regione"] = df["Città"].map(regioni)
df

In [ ]:
def fascia_eta(eta):
    """Restituisce la fascia d'età di una persona."""
    if eta < 25:
        return "under 25"
    return "25 e oltre"


df["Fascia"] = df["Età"].apply(fascia_eta)
df

<a id="sez-2-4"></a>
### 2.4 Gestione degli indici

Dopo un ordinamento con `sort_values` (lo vediamo tra poco) le righe tengono l'indice di prima,
in disordine. `reset_index(drop=True)` lo fa ripartire da 0 e scarta il vecchio.

In [ ]:
df.sort_values("Età").reset_index(drop=True)

<a id="sez-2-5"></a>
### 2.5 Rinomina delle colonne

Possiamo rinominare le colonne con `rename` e un dizionario da nome vecchio a nome nuovo.

In [ ]:
df_rinominato = df.rename(columns={"Città": "Residenza", "Età": "age"})
df_rinominato

<a id="sez-3"></a>
## 3. Ordinamento dei dati

L'ordinamento serve sia nell'analisi sia nella presentazione dei risultati.

<a id="sez-3-1"></a>
### 3.1 sort_values e sort_index

`sort_values()` ordina i dati in base ai valori di una o più colonne; `ascending=False` ordina
dal più grande.

In [ ]:
# ordinare per età
df_ordinato = df.sort_values(by="Età", ascending=False)
df_ordinato

`sort_index()` ordina in base all'indice: è utile quando l'indice ha un significato.

<a id="sez-3-2"></a>
### 3.2 Ordinamento su più colonne

Possiamo ordinare usando più colonne come chiavi: a pari valore della prima decide la seconda.
`ascending` accetta una lista, un valore per colonna.

In [ ]:
# ordinare per età e poi per nome
df_ordinato_2 = df.sort_values(by=["Età", "Nome"], ascending=[False, True])
df_ordinato_2

<a id="sez-4"></a>
## 4. Groupby

Il metodo `groupby()` **suddivide** un DataFrame in gruppi in base a una o più colonne, per poi
applicare un'**aggregazione** ai dati di ciascun gruppo. È la logica **Split-Apply-Combine**:

1. **Split**: dividere i dati in gruppi in base a una o più colonne.
2. **Apply**: applicare una funzione (somma, media, conteggio) a ogni gruppo.
3. **Combine**: unire i risultati in un nuovo oggetto pandas (Series o DataFrame).

La forma è `df.groupby("colonna_di_raggruppamento")["colonna"].funzione()`.

In [ ]:
vendite = pd.DataFrame({
    "Categoria": ["A", "B", "A", "B", "A", "C"],
    "Vendite": [100, 200, 150, 250, 120, 300],
})

# raggruppiamo per Categoria e sommiamo le vendite
vendite.groupby("Categoria")["Vendite"].sum()

`groupby()` da solo restituisce un oggetto GroupBy, che aspetta una funzione per elaborare i
gruppi. Con `sum()` otteniamo una riga per gruppo, con la categoria nell'indice.

<a id="sez-5"></a>
## 5. Unione e concatenazione di DataFrame

Pandas offre diversi modi per combinare più DataFrame.

<a id="sez-5-1"></a>
### 5.1 Merge

`merge()` combina DataFrame in base a chiavi comuni, come un join in SQL. Tipi di join:

- **Inner join**: le righe con chiavi presenti in entrambi i DataFrame.
- **Left join**: tutte le righe del DataFrame di sinistra e quelle corrispondenti di destra.
- **Right join**: tutte le righe del DataFrame di destra e quelle corrispondenti di sinistra.
- **Outer join**: tutte le righe, che abbiano una corrispondenza o no.

In [ ]:
# DataFrame di esempio
df_sinistra = pd.DataFrame({"ID": [1, 2, 3], "Nome": ["Alice", "Bob", "Charlie"]})
df_destra = pd.DataFrame({"ID": [3, 4, 5], "Età": [35, 40, 45]})

# inner join
df_inner = pd.merge(df_sinistra, df_destra, on="ID", how="inner")
df_inner

In [ ]:
# outer join
df_outer = pd.merge(df_sinistra, df_destra, on="ID", how="outer")
df_outer

Con il left join teniamo tutte le righe del DataFrame di sinistra: dove manca la corrispondenza
resta `NaN`.

In [ ]:
# left join
pd.merge(df_sinistra, df_destra, on="ID", how="left")

<a id="sez-5-2"></a>
### 5.2 Concatenazione

`concat()` unisce DataFrame verticalmente, aggiungendo righe.

In [ ]:
# concatenazione verticale
df_concatenato = pd.concat([df_sinistra, df_destra], ignore_index=True, sort=False)
df_concatenato

`merge` unisce su una chiave comune, `concat` mette una tabella sotto l'altra.

Riassunto in una pagina: [Da Excel a pandas](../Schede/Scheda_Excel_pandas.md).

<a id="sez-6"></a>
## 6. Esercizi

Lavori come data analyst per un'azienda che gestisce una piattaforma di streaming musicale.
Il dataset con gli ascolti degli utenti è questo: eseguilo prima dei primi tre esercizi.

In [ ]:
# DataFrame degli ascolti
dati_ascolti = {
    "UserID": [101, 102, 103, 104, 105, 106],
    "Song": ["Song A", "Song B", "Song A", "Song C", "Song B", "Song D"],
    "Artist": ["Artist X", "Artist Y", "Artist X", "Artist Z", "Artist Y", "Artist W"],
    "Plays": [15, 2, 4, 1, 5, 2],
}
ascolti = pd.DataFrame(dati_ascolti)
ascolti

<a id="es-7-1"></a>
<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Esercizio 7.1 · Gli ascolti di un utente**

**Cosa fare.**

Trova tutte le canzoni ascoltate dall'utente con `UserID` 103: filtra `ascolti` per tenere
solo le righe dove `UserID` è 103 e salva il risultato in `ascolti_utente_103`.

Output atteso: una riga, `Song A` di `Artist X` con 4 ascolti.

*Suggerimento:* usa il filtraggio condizionale con una maschera booleana.

</div>

<div style="background:rgba(46,159,214,0.12); border-left:4px solid #2e9fd6; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✅ Soluzione 7.1**

</div>

In [ ]:
cond = ascolti["UserID"] == 103
ascolti_utente_103 = ascolti[cond]
ascolti_utente_103

In [ ]:
from corso import verifica
verifica("7.1")

<a id="es-7-2"></a>
<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Esercizio 7.2 · Totale di ascolti per canzone**

**Cosa fare.**

Calcola il totale di ascolti per ogni canzone: raggruppa per `Song` e somma i valori di
`Plays`. Salva il risultato in `ascolti_per_canzone`.

Output atteso: Song A 19, Song B 7, Song C 1, Song D 2.

*Suggerimento:* usa `groupby()` seguito da `sum()`.

</div>

<div style="background:rgba(46,159,214,0.12); border-left:4px solid #2e9fd6; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✅ Soluzione 7.2**

</div>

In [ ]:
ascolti_per_canzone = ascolti.groupby("Song")["Plays"].sum()
ascolti_per_canzone

In [ ]:
from corso import verifica
verifica("7.2")

<a id="es-7-3"></a>
<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Esercizio 7.3 · L'artista più ascoltato**

**Cosa fare.**

Identifica l'artista più popolare: raggruppa per `Artist`, somma `Plays` e ordina in modo
decrescente (`ascolti_per_artista`). Poi salva in `artista_top` il nome dell'artista con il
totale di ascolti più alto.

*Suggerimento:* combina `groupby()`, `sum()` e `sort_values()`; `idxmax()` restituisce l'etichetta del valore massimo.

</div>

<div style="background:rgba(46,159,214,0.12); border-left:4px solid #2e9fd6; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✅ Soluzione 7.3**

</div>

In [ ]:
ascolti_per_artista = ascolti.groupby("Artist")["Plays"].sum().sort_values(ascending=False)
artista_top = ascolti_per_artista.idxmax()
artista_top

In [ ]:
from corso import verifica
verifica("7.3")

<a id="es-7-4"></a>
<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Esercizio 7.4 · Prezzo medio dell'elettricità per stato (facoltativo)**

Il file `U.S. Electricity Prices.csv` contiene il prezzo medio mensile dell'elettricità negli
Stati Uniti (in centesimi di dollaro per kWh), per stato e per settore, dal 2001 al 2024.

**Cosa fare.**

1. Leggi il file in `prezzi` e tieni solo le righe con `sectorName` uguale a `"all sectors"`.
2. Calcola il prezzo medio (`price`) per stato (`stateDescription`) in `prezzo_medio`,
   con `reset_index()` per riavere lo stato come colonna.
3. Unisci `stati` (già definito nella cella) e `prezzo_medio` con un inner join sulla
   colonna `stateDescription`, nel DataFrame `prezzi_stati`.

Output atteso: quattro righe; il Texas ha un prezzo medio di circa 8,86.

*Suggerimento:* guarda prima le colonne con `prezzi.head()`.

</div>

<div style="background:rgba(46,159,214,0.12); border-left:4px solid #2e9fd6; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✅ Soluzione 7.4**

Il file contiene anche regioni e il totale nazionale (`New England`, `U.S. Total`): l'inner
join con `stati` tiene solo gli stati che ci interessano, con la loro sigla.

</div>

In [ ]:
prezzi = pd.read_csv("../Dati/U.S. Electricity Prices.csv")
stati = pd.DataFrame({
    "stateDescription": ["California", "Texas", "New York", "Florida"],
    "sigla": ["CA", "TX", "NY", "FL"],
})

tutti_settori = prezzi[prezzi["sectorName"] == "all sectors"]
prezzo_medio = tutti_settori.groupby("stateDescription")["price"].mean().reset_index()
prezzi_stati = pd.merge(stati, prezzo_medio, on="stateDescription", how="inner")
prezzi_stati

In [ ]:
from corso import verifica
verifica("7.4")

---

**Fine del notebook 07.** Precedente: [E2 · Esercitazione 2](Esercitazione_2.ipynb) &nbsp;·&nbsp; Prossimo: [08 · Pandas: le date](08_Pandas_date.ipynb)

<sub>Blocco 3 · Analisi dei dati con pandas e visualizzazione · Giornata 2 · Aula Base</sub>